In [1]:
# 01 패키지
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
# 02 랜덤 고정값
np.random.seed(42)
random.seed(42)

In [3]:
# 03 경로 설정
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "11_influencer_campaign_review.csv"

In [4]:
# 04 campaigns 데이터 불러오기
campaigns = pd.read_csv(DATA_DIR / "02_campaigns.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
campaigns.columns = campaigns.columns.str.strip()

# 빈 문자열을 NaN으로 변환
campaigns = campaigns.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
campaigns = campaigns.dropna(how="all")

# campaign_id가 없는 행 제거
campaigns = campaigns.dropna(subset=["campaign_id"])

# 인덱스 재정렬
campaigns = campaigns.reset_index(drop=True)

# 날짜 컬럼 변환
campaigns["campaign_start_at"] = pd.to_datetime(campaigns["campaign_start_at"], errors="coerce")
campaigns["campaign_end_at"] = pd.to_datetime(campaigns["campaign_end_at"], errors="coerce")

# 데이터 확인
print(campaigns.shape)
campaigns.head()

(3, 6)


,campaign_id,client_id,campaign_name,campaign_start_at,campaign_end_at,campaign_status
0,cam-0001,cli-0001,[라이언스윔] ROAR 남성 수영복 세트 (수영모+수영복+수경) 체험단 모집,2026-05-16 10:00:00,2026-05-30 23:59:00,종료
1,cam-0002,cli-0001,[라이언스윔] ROAR 여성 수영복 세트 (수영모+수영복+수경) 체험단 모집,2026-05-16 10:00:00,2026-05-30 23:59:00,종료
2,cam-0003,cli-0001,[라이언스윔] ROAR 수영 샴푸,2026-05-29 10:00:00,2026-06-12 23:59:00,종료


In [5]:
# 05 influencers 데이터 불러오기
influencers = pd.read_csv(DATA_DIR / "04_influencers.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
influencers.columns = influencers.columns.str.strip()

# 빈 문자열을 NaN으로 변환
influencers = influencers.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
influencers = influencers.dropna(how="all")

# influencer_id가 없는 행 제거
influencers = influencers.dropna(subset=["influencer_id"])

# 인덱스 재정렬
influencers = influencers.reset_index(drop=True)

# 데이터 확인
print(influencers.shape)
influencers.head()

(200, 12)


,influencer_id,influencer_name,influencer_created_at,influencer_registration_type,influencer_platform,influencer_category,influencer_followers_count,influencer_posts,influencer_avg_likes,influencer_avg_comments,influencer_engagement_rate,influencer_recent_post_at
0,Inf-0001,zrathbourne0,2025-12-02 09:57:47,스포츠크루 및 커뮤니티,Instagram,바레,65010,71,302.49,241.75,0.84,2026-02-17 11:09:28
1,Inf-0002,kalred1,2025-10-04 15:03:03,개인 인플루언서,Instagram,크로스핏,90294,159,805.09,331.81,1.26,2026-04-12 02:27:17
2,Inf-0003,lelgey2,2025-07-03 21:59:09,웰니스 피트니스 센터,YouTube,요가,75232,250,239.19,228.80,0.62,2026-02-17 09:38:13
3,Inf-0004,hcasson3,2025-10-25 05:41:08,웰니스 피트니스 센터,YouTube,요가,53117,270,405.20,178.78,1.10,2026-02-26 01:28:20
4,Inf-0005,lmangam4,2025-11-11 03:06:02,스포츠크루 및 커뮤니티,YouTube,러닝,55393,221,619.50,528.73,2.07,2026-04-10 05:26:51


In [6]:
# 06 campaign_influencers 데이터 불러오기

campaign_influencers = pd.read_csv(DATA_DIR / "18_campaign_influencers.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
campaign_influencers.columns = campaign_influencers.columns.str.strip()

# 빈 문자열을 NaN으로 변환
campaign_influencers = campaign_influencers.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
campaign_influencers = campaign_influencers.dropna(how="all")

# 필수 ID 없는 행 제거
campaign_influencers = campaign_influencers.dropna(
    subset=["campaign_influencer_id", "campaign_id", "influencer_id", "participation_status"]
)

# 날짜 컬럼 변환
campaign_influencers["applied_at"] = pd.to_datetime(campaign_influencers["applied_at"], errors="coerce")
campaign_influencers["selected_at"] = pd.to_datetime(campaign_influencers["selected_at"], errors="coerce")

# 인덱스 재정렬
campaign_influencers = campaign_influencers.reset_index(drop=True)

# 데이터 확인
print(campaign_influencers.shape)
campaign_influencers.head()

(79, 7)


,campaign_influencer_id,influencer_id,campaign_id,applied_at,selected_at,participation_status,reward_amount
0,caminf-0001,Inf-0059,cam-0001,2026-05-19 12:17:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
1,caminf-0002,Inf-0041,cam-0001,2026-05-17 14:42:00,NaT,지원,0
2,caminf-0003,Inf-0035,cam-0001,2026-05-19 03:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
3,caminf-0004,Inf-0103,cam-0001,2026-05-18 21:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
4,caminf-0005,Inf-0185,cam-0001,2026-05-18 13:19:00,2026-05-23 10:00:00,선정,0


In [7]:
# 07 참여 완료 인플루언서만 필터링

completed_campaign_influencers = campaign_influencers[
    campaign_influencers["participation_status"] == "콘텐츠 등록 완료"
].copy()

completed_campaign_influencers = completed_campaign_influencers.reset_index(drop=True)

print("콘텐츠 등록 완료 행 수:", len(completed_campaign_influencers))
completed_campaign_influencers.head()

콘텐츠 등록 완료 행 수: 27


,campaign_influencer_id,influencer_id,campaign_id,applied_at,selected_at,participation_status,reward_amount
0,caminf-0001,Inf-0059,cam-0001,2026-05-19 12:17:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
1,caminf-0003,Inf-0035,cam-0001,2026-05-19 03:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
2,caminf-0004,Inf-0103,cam-0001,2026-05-18 21:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
3,caminf-0014,Inf-0090,cam-0001,2026-05-20 10:17:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000
4,caminf-0015,Inf-0111,cam-0001,2026-05-20 20:51:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000


In [8]:
# 08 campaigns와 결합해서 campaign_start_at 가져오기

review_base = completed_campaign_influencers.merge(
    campaigns[["campaign_id", "campaign_start_at", "campaign_end_at"]],
    on="campaign_id",
    how="left"
)

# influencer_id가 실제 influencers 테이블에 존재하는지 확인하기 위해 결합
review_base = review_base.merge(
    influencers[["influencer_id"]],
    on="influencer_id",
    how="left"
)

# campaign_start_at 없는 행 확인
print("campaign_start_at 누락:", review_base["campaign_start_at"].isna().sum())

# influencer_id 누락 확인
print("influencer_id 누락:", review_base["influencer_id"].isna().sum())

review_base.head()

campaign_start_at 누락: 0
influencer_id 누락: 0


,campaign_influencer_id,influencer_id,campaign_id,applied_at,selected_at,participation_status,reward_amount,campaign_start_at,campaign_end_at
0,caminf-0001,Inf-0059,cam-0001,2026-05-19 12:17:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000,2026-05-16 10:00:00,2026-05-30 23:59:00
1,caminf-0003,Inf-0035,cam-0001,2026-05-19 03:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000,2026-05-16 10:00:00,2026-05-30 23:59:00
2,caminf-0004,Inf-0103,cam-0001,2026-05-18 21:29:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000,2026-05-16 10:00:00,2026-05-30 23:59:00
3,caminf-0014,Inf-0090,cam-0001,2026-05-20 10:17:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000,2026-05-16 10:00:00,2026-05-30 23:59:00
4,caminf-0015,Inf-0111,cam-0001,2026-05-20 20:51:00,2026-05-23 10:00:00,콘텐츠 등록 완료,20000,2026-05-16 10:00:00,2026-05-30 23:59:00


In [9]:
# 09 influencer_campaign_review 데이터 생성

contact_channel_list = ["Instagram DM", "Email", "KakaoTalk", "Message"]

review_rows = []
review_id = 1

for _, row in review_base.iterrows():
    campaign_id = row["campaign_id"]
    influencer_id = row["influencer_id"]
    campaign_start_at = row["campaign_start_at"]

    # 연락 일시
    # 캠페인 시작일 기준 0 ~ 6일 전부터 캠페인 시작 후 2일 사이 날짜 생성
    contact_start = campaign_start_at - timedelta(days=21)
    contact_end = campaign_start_at + timedelta(days=2)

    random_seconds = np.random.randint(
        0,
        int((contact_end - contact_start).total_seconds()) + 1
    )

    contacted_at = contact_start + timedelta(seconds=int(random_seconds))

    # 연락 채널
    contact_channel = random.choice(contact_channel_list)

    # 캠페인 협업 평점
    # 실제 참여 완료자 대상 후기이므로 3~5점이 많이 나오도록 설정
    campaign_review_score = np.random.choice(
        [1, 2, 3, 4, 5],
        p=[0.03, 0.07, 0.25, 0.40, 0.25]
    )

    review_rows.append({
        "influencer_campaign_review_id": f"infcam-{review_id:04d}",
        "influencer_id": influencer_id,
        "campaign_id": campaign_id,
        "contacted_at": contacted_at.strftime("%Y-%m-%d %H:%M:%S"),
        "contact_channel": contact_channel,
        "campaign_review_score": campaign_review_score
    })

    review_id += 1

influencer_campaign_review = pd.DataFrame(review_rows)

print(influencer_campaign_review.shape)
influencer_campaign_review.head()

(27, 6)


,influencer_campaign_review_id,influencer_id,campaign_id,contacted_at,contact_channel,campaign_review_score
0,infcam-0001,Inf-0059,cam-0001,2026-04-26 19:52:38,Instagram DM,5
1,infcam-0002,Inf-0035,cam-0001,2026-05-11 18:53:34,Instagram DM,4
2,infcam-0003,Inf-0103,cam-0001,2026-04-26 16:37:48,KakaoTalk,4
3,infcam-0004,Inf-0090,cam-0001,2026-04-27 00:08:57,Email,3
4,infcam-0005,Inf-0111,cam-0001,2026-05-08 13:34:34,Email,5


In [10]:
# 08 CSV 저장

influencer_campaign_review.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: ..\data\11_influencer_campaign_review.csv
